In [1]:
import h5py
import os
import numpy as np
import json
import shutil
import cv2
from PIL import Image

In [ ]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/jones_golden_idol"
config_dict = {}
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)

with h5py.File(config_dict["original_file"], "r") as demo_file:
    rgb = demo_file["data/human_demo/obs/agentview_rgb/"][()]
    data_config = json.loads(demo_file["data"].attrs['data_config'])
    K = data_config["intrinsics"]["camera_rs_0"]
masks = np.load(os.path.join(annotation_folder, "masks.npz"))['arr_0']
print(rgb.shape)
print(data_config)
print(masks.shape)

idx = 0
target_ob = 1
masks[masks != target_ob] = 0
masks[masks == target_ob] = 255
display(Image.fromarray(masks[idx]))
display(Image.fromarray((rgb[idx])))

rgb_target = Image.fromarray(rgb[idx])
mask_target = Image.fromarray(masks[idx]).convert('L')

image_with_transparency = np.dstack((rgb_target, mask_target))



In [ ]:
from pathlib import Path
img_folder = annotation_folder.split("/")[-1]
mesh_img_folder = Path("/home/arisrei/research/RPL/ORION/images_for_meshes")
print(f"saving to {os.path.join(mesh_img_folder, img_folder, f'{img_folder}.png')}")
print(f"saving to {os.path.join(mesh_img_folder, img_folder, f'{img_folder}_with_bg.png')}")


#recenter the image by cropping
mask = np.array(mask_target)
nonzero = np.nonzero(mask)
padding = np.zeros(2, dtype=int)
y1 = max(0, np.min(nonzero[0]) - padding[0])
y2 = min(np.max(nonzero[0]) + padding[0], mask.shape[0])
x1 = max(0, np.min(nonzero[1]) - padding[1])
x2 = min(np.max(nonzero[1]) + padding[1], mask.shape[1])
print(f"({y1}, {x1}) ({y2}, {x2})")

recentered_mask = Image.fromarray(mask[y1:y2, x1:x2])
recentered_rgb = Image.fromarray(rgb[idx][y1:y2, x1:x2, :])

display(Image.fromarray(image_with_transparency))#.save('image_trs.png')

print('recentered rgb')
display(recentered_rgb)
recentered_image_with_transparency = np.dstack((recentered_rgb, recentered_mask))

print('recentered_image_with_transparency')
display(Image.fromarray(recentered_image_with_transparency))#.save('image_trs.png')

os.makedirs(os.path.join(mesh_img_folder, img_folder), exist_ok=True)
Image.fromarray(recentered_image_with_transparency).save(os.path.join(mesh_img_folder, img_folder, f"{img_folder}.png"))
recentered_rgb.save(os.path.join(mesh_img_folder, img_folder, f"{img_folder}_with_bg.png"))

